<a href="https://colab.research.google.com/github/Gunnny/dream-atlas/blob/main/notebooks/01_explore_data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
!pip install datasets

In [10]:
from datasets import load_dataset
ds = load_dataset("DReAMy-lib/DreamBank-dreams-en")
print(ds)

README.md:   0%|          | 0.00/2.45k [00:00<?, ?B/s]

data/train-00000-of-00001-24937aef854be1(…): reconstructing file:   0%|          |  0.00B / 12.0MB            

data/train-00000-of-00001-24937aef854be1(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/22415 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['series', 'description', 'dreams', 'gender', 'year'],
        num_rows: 22415
    })
})


In [11]:
import pandas as pd

df = ds["train"].to_pandas()
df["word_count"] = df["dreams"].str.split().str.len()

df_clean = df[df["word_count"] >= 5].copy()
print(len(df), "→", len(df_clean))

print("vorher:", len(df_clean))
df_clean = df_clean.drop_duplicates(subset="dreams").reset_index(drop=True)
print("nachher:", len(df_clean))

df_clean.to_parquet("dreams_clean.parquet")

22415 → 22347
vorher: 22347
nachher: 22096


Nur Berichte mit unter 5 Wörtern entfernt (68 Zeilen, 0,3 %) – das sind Titel ohne Text. Kurze Berichte ab 5 Wörtern sind inhaltlich vollständig und bleiben erhalten; gefiltert wird später pro Anwendungsfall über word_count.

In [12]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(stop_words="english", min_df=2)
X = vectorizer.fit_transform(df_clean["dreams"])
print(X.shape)

zellen = X.shape[0] * X.shape[1]
print("Zellen insgesamt:", zellen)
print("davon besetzt:   ", X.nnz)
print("Füllgrad:        ", round(100 * X.nnz / zellen, 3), "%")
print("Speicher:        ", round(X.data.nbytes / 1e6, 1), "MB")
print("wäre dicht:      ", round(zellen * 8 / 1e9, 2), "GB")

(22096, 24011)
Zellen insgesamt: 530547056
davon besetzt:    1170708
Füllgrad:         0.221 %
Speicher:         9.4 MB
wäre dicht:       4.24 GB


In [13]:
import numpy as np

def search(query, k=5):
    q = vectorizer.transform([query])
    sims = (X @ q.T).toarray().ravel()
    top = np.argsort(-sims)[:k]

    for i in top:
        print(round(float(sims[i]), 3), "|", df_clean["dreams"].iloc[i][:200])
        print("---")


0.415 | Nightmare. I wake up frightened. I was being chased.
---
0.391 | I was being chased by two people. It was a man and a woman. They chased me into a house where a stranger was sitting at a desk trying to make a computer run on old batteries. I went to find some and w
---
0.371 | It was a dream similar to the one on the 6th of September; the same familiar characters (schoolmates) and others I have never seen or I don't remember. My dream was still centered around the girl I lo
---
0.349 | Another dream I had was that I was being chased by government agents because they had done certain things that I was against.
---
0.325 | I and several other people -- I don't believe I knew any of these people in particular, but they were all about my age ---were being chased by this group of other people, some of them older people, so
---
0.295 | Through The Bowling Alley  I'm at Carter School. I'm leading a group of younger kids. We're being pursued. We get out of the school through the bowlin

In [22]:
search("I was being chased through a dark house")
search("I was being pursued through a dark house")
search("Ich wurde durch ein dunkles Haus verfolgt")

0.415 | Nightmare. I wake up frightened. I was being chased.
---
0.391 | I was being chased by two people. It was a man and a woman. They chased me into a house where a stranger was sitting at a desk trying to make a computer run on old batteries. I went to find some and w
---
0.371 | It was a dream similar to the one on the 6th of September; the same familiar characters (schoolmates) and others I have never seen or I don't remember. My dream was still centered around the girl I lo
---
0.349 | Another dream I had was that I was being chased by government agents because they had done certain things that I was against.
---
0.325 | I and several other people -- I don't believe I knew any of these people in particular, but they were all about my age ---were being chased by this group of other people, some of them older people, so
---
0.295 | Through The Bowling Alley  I'm at Carter School. I'm leading a group of younger kids. We're being pursued. We get out of the school through the bowlin

Baseline TF-IDF. Lexikalische Suche funktioniert bei wörtlicher Übereinstimmung (bester Treffer 0,413). Beim Synonym „pursued“ statt „chased“ fällt der beste Wert auf 0,293, und die inhaltlich passenden Treffer aus der ersten Anfrage erscheinen gar nicht mehr. Bei deutscher Anfrage versagt das Verfahren vollständig (Höchstwert 0,27, Treffer rein zufällig über Ländernamen). Grund: TF-IDF vergleicht Zeichenketten, keine Bedeutung.

In [14]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "keine GPU")

True
Tesla T4


In [15]:
!pip install -q sentence-transformers


In [16]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")
print(model.get_sentence_embedding_dimension())


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

384


/tmp/ipykernel_402/850366859.py:4: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(model.get_sentence_embedding_dimension())


Batches:   0%|          | 0/346 [00:00<?, ?it/s]

(22096, 384)


In [17]:
texte = df_clean["dreams"].tolist()

E = model.encode(
    texte,
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True,
)
print(E.shape)

Batches:   0%|          | 0/346 [00:00<?, ?it/s]

(22096, 384)


In [18]:
print(len(df_clean), X.shape[0], E.shape[0])

22096 22096 22096


In [25]:
def search_emb(query, k=5, min_words=20):
    q = model.encode([query], normalize_embeddings=True)
    sims = (E @ q.T).ravel()

    maske = df_clean["word_count"].to_numpy() >= min_words
    sims = np.where(maske, sims, -1)

    top = np.argsort(-sims)[:k]
    for i in top:
        print(round(float(sims[i]), 3), "|", df_clean["dreams"].iloc[i][:200])
        print("---")

In [26]:
search_emb("I was being chased through a dark house")
search_emb("I was being pursued through a dark house")
search_emb("Ich wurde durch ein dunkles Haus verfolgt")

0.675 | Someone is trying to kill me. I was in my house and it actually looked like my house did about 7or 8 years ago. In the dream I wake up, realized there were people trying to break into my house. I ran 
---
0.668 | I was at the door of a dark room. I heard someone inside the room call me in. I was afraid and I backed away. But then I felt a hand pulling me inside, and a feeling of panic and terror came over me. 
---
0.667 | I think it was nighttime. I went into a hallway. The door at the end of the hall was closed. I thought I saw a shadow through the crack at the bottom of the door. I hesitated but decided to open it ju
---
0.665 | I ran all the way home, facing traffic, along a busy street lined with trees, dodging in and out among the trunks and branches, pretending I was catching passes, evading tacklers, etc.  My neighborhoo
---
0.66 | I was walking home down a country road at dusk when an unknown man began chasing me. I was terrified and ran furiously toward my home. I mana

In [29]:
queries = [
    # --- Sprachpaare: inhaltlich identisch, DE/EN ---
    {"id": "zaehne_de", "lang": "de", "topic": "teeth",
     "text": "Meine Zähne fielen mir einer nach dem anderen aus und ich konnte nichts dagegen tun"},
    {"id": "zaehne_en", "lang": "en", "topic": "teeth",
     "text": "My teeth were falling out one by one and I could not stop it"},

    {"id": "fliegen_de", "lang": "de", "topic": "flying",
     "text": "Ich konnte fliegen und bin über die Dächer der Stadt geschwebt"},
    {"id": "fliegen_en", "lang": "en", "topic": "flying",
     "text": "I could fly and floated over the rooftops of the city"},

    {"id": "spaet_de", "lang": "de", "topic": "late",
     "text": "Ich kam zu spät zu einer wichtigen Prüfung und konnte den Raum nicht finden"},
    {"id": "spaet_en", "lang": "en", "topic": "late",
     "text": "I was late for an important exam and could not find the room"},

    # --- Deutsch ---
    {"id": "entfuehrung", "lang": "de", "topic": "kidnapping",
     "text": "Zwei Männer wollten mich entführen und in ein Auto zerren"},
    {"id": "schwimmen", "lang": "de", "topic": "swimming",
     "text": "Ich bin in einem See geschwommen und kam nicht mehr ans Ufer zurück"},
    {"id": "klauen", "lang": "de", "topic": "stealing",
     "text": "Ich habe in einem Laden etwas gestohlen und hatte Angst, erwischt zu werden"},
    {"id": "messer", "lang": "de", "topic": "knife",
     "text": "Ein fremder Mann stand vor mir und bedrohte mich mit einem Messer"},

    # --- Englisch ---
    {"id": "tod", "lang": "en", "topic": "death",
     "text": "My grandmother who died years ago was alive again and talking to me"},
    {"id": "liebe", "lang": "en", "topic": "love",
     "text": "I was in love with someone and we were happy together"},
    {"id": "suche", "lang": "en", "topic": "searching",
     "text": "I was looking for something but I could not remember what it was"},
    {"id": "militaer", "lang": "en", "topic": "military",
     "text": "I was a soldier in a war and there was shooting all around me"},
    {"id": "drogen", "lang": "en", "topic": "drugs",
     "text": "I had taken some kind of drug and everything around me felt strange"},
    {"id": "sex", "lang": "en", "topic": "sex",
     "text": "I was in bed being intimate with someone I knew"},
    {"id": "verfolgung", "lang": "en", "topic": "chase",
     "text": "I was being chased through a dark house"},

    # --- Härtefälle: vermutlich nicht im Korpus ---
    {"id": "gaming", "lang": "en", "topic": "gaming",
     "text": "I was playing a video game and could not stop playing"},
    {"id": "drachen", "lang": "en", "topic": "dragons",
     "text": "A dragon was flying over a castle and breathing fire"},
    {"id": "mathe", "lang": "de", "topic": "math",
     "text": "Ich musste an der Tafel eine schwierige Matheaufgabe lösen"},
]

print(len(queries))


20


In [30]:
def retrieve_tfidf(query, k=5):
    q = vectorizer.transform([query])
    sims = (X @ q.T).toarray().ravel()
    return np.argsort(-sims)[:k]

def retrieve_emb(query, k=5, min_words=0):
    q = model.encode([query], normalize_embeddings=True)
    sims = (E @ q.T).ravel()
    if min_words > 0:
        maske = df_clean["word_count"].to_numpy() >= min_words
        sims = np.where(maske, sims, -1)
    return np.argsort(-sims)[:k]

In [31]:
from collections import defaultdict

methoden = ["tfidf", "emb", "emb20"]
runs = {}
pool = defaultdict(set)

for qu in queries:
    runs[(qu["id"], "tfidf")] = list(retrieve_tfidf(qu["text"]))
    runs[(qu["id"], "emb")]   = list(retrieve_emb(qu["text"], min_words=0))
    runs[(qu["id"], "emb20")] = list(retrieve_emb(qu["text"], min_words=20))

    for m in methoden:
        pool[qu["id"]].update(runs[(qu["id"], m)])

print("Paare zu bewerten:", sum(len(v) for v in pool.values()))

Paare zu bewerten: 213


In [32]:
rows = []
for qu in queries:
    for idx in sorted(pool[qu["id"]]):
        rows.append({
            "query_id": qu["id"],
            "query":    qu["text"],
            "doc_idx":  int(idx),
            "dream":    df_clean["dreams"].iloc[idx],
            "relevant": "",
        })

judge = (pd.DataFrame(rows)
         .sample(frac=1, random_state=42)
         .sort_values("query_id", kind="stable")
         .reset_index(drop=True))

judge.to_csv("judgements.csv", index=False)
print(len(judge))

213


**Relevant** = 1, wenn das zentrale Ereignis der Anfrage im Traum tatsächlich vorkommt.

**Nicht relevant** = 0, wenn nur die Stimmung oder das Umfeld passt.

**Beispiel**: Anfrage „verfolgt werden“. Ein Traum über Flucht vor einem Mann → 1. Ein Traum über ein dunkles Haus ohne Verfolgung → 0.

In [35]:
gold = pd.read_csv("judgements_final.csv")
lookup = {(r.query_id, r.doc_idx): r.relevant for r in gold.itertuples()}

ergebnis = {}
for m in methoden:
    p = [sum(lookup.get((qu["id"], int(i)), 0) for i in runs[(qu["id"], m)]) / 5
         for qu in queries]
    ergebnis[m] = sum(p) / len(p)

for m, v in ergebnis.items():
    print(f"{m:8s} P@5 = {v:.3f}")

tfidf    P@5 = 0.360
emb      P@5 = 0.560
emb20    P@5 = 0.580


In [36]:
for qu in queries:
    a = sum(lookup.get((qu["id"], int(i)), 0) for i in runs[(qu["id"], "emb")]) / 5
    b = sum(lookup.get((qu["id"], int(i)), 0) for i in runs[(qu["id"], "emb20")]) / 5
    if a != b:
        print(f'{qu["id"]:12s} emb={a:.1f}  emb20={b:.1f}')

fliegen_de   emb=0.6  emb20=0.8
schwimmen    emb=0.2  emb20=0.6
klauen       emb=0.4  emb20=0.6
tod          emb=0.4  emb20=0.2
militaer     emb=0.4  emb20=0.2
sex          emb=0.4  emb20=0.2
gaming       emb=0.4  emb20=0.2
drachen      emb=0.6  emb20=0.8
mathe        emb=0.4  emb20=0.6


In [37]:
for lang in ["de", "en"]:
    ids = [qu["id"] for qu in queries if qu["lang"] == lang]
    p = [sum(lookup.get((qid, int(i)), 0) for i in runs[(qid, "emb")]) / 5 for qid in ids]
    print(lang, len(ids), round(sum(p)/len(p), 3))

de 8 0.55
en 12 0.567
